# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [4]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

#REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
REPO_ROOT = "/home/tlht/day01/K4-Track4-Day1-TranCongThien-02A202602579-Neuralnetwork"
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch version:", torch.__version__)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


PyTorch version: 2.14.0+cu130
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [5]:
# Chạy scripts/split_data.py
subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

print(f"X_tr shape: {data['X_tr'].shape}")
print(f"X_val shape: {data['X_val'].shape}")
print(f"X_eval shape: {data['X_eval'].shape}")

# Kiểm tra mean, std của 10 cột đầu tập train
numeric_mean = data['X_tr'][:, :10].mean(dim=0)
numeric_std = data['X_tr'][:, :10].std(dim=0)
print(f"Numeric columns mean: {numeric_mean.abs().max().item():.4f} (expect ~0)")
print(f"Numeric columns std: {numeric_std.mean().item():.4f} (expect ~1)")


train: X (464809, 54) float32, y (464809,) int64, nhãn 0..6 -> data/processed/train.npz
eval : X (116203, 54) float32, y (116203,) int64, nhãn 0..6 -> data/processed/eval.npz

lớp   train(%)   eval(%)
  0    36.461    36.460
  1    48.760    48.760
  2     6.154     6.154
  3     0.473     0.472
  4     1.634     1.634
  5     2.989     2.989
  6     3.530     3.530

đoán luôn lớp đa số (lớp 1) cho accuracy trên eval = 0.4876
Train size: 371847, Val size: 92962, Eval size: 116203
Majority class on train: 1, Val majority accuracy: 0.4876
X_tr shape: torch.Size([371847, 54])
X_val shape: torch.Size([92962, 54])
X_eval shape: torch.Size([116203, 54])
Numeric columns mean: 0.0003 (expect ~0)
Numeric columns std: 1.0000 (expect ~1)


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [16]:
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256,128)], "Số tham số không khớp"
print("Số tham số:", count_params(model))

dummy_x = torch.randn(8, 54, device=device)
logits = model(dummy_x)
print("Logits shape:", logits.shape)
assert logits.shape == (8, 7), "Shape logits sai"

# Mất mát bước 0
import torch.nn.functional as F
with torch.no_grad():
    loss_step0 = F.cross_entropy(logits, torch.zeros(8, dtype=torch.long, device=device))
print("Loss ngẫu nhiên:", loss_step0.item(), "Kỳ vọng:", np.log(7))


Số tham số: 47879
Logits shape: torch.Size([8, 7])
Loss ngẫu nhiên: 1.9433746337890625 Kỳ vọng: 1.9459101490553132


**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

loss bước 0 đo được = 2.744, cao hơn so với kỳ vọng (hoàn toàn ngẫu nhiên và đồng đều) là ln(7) $\approx$ 1.946.
Vì sao: Mức loss $\ln(7)$ chỉ đạt được khi mô hình dự đoán xác suất cho 7 lớp là bằng nhau hoàn toàn ($p = 1/7$). Tuy nhiên, ở đây mô hình đang sử dụng khởi tạo trọng số He initialization (init="he"). Khởi tạo này làm cho các giá trị ở lớp đầu ra (logits) ban đầu có phương sai khá lớn. Điều này khiến cho phân bố xác suất qua hàm Softmax bị chênh lệch (có lớp rất cao, lớp rất thấp) chứ không đồng đều, dẫn đến Cross-Entropy loss ban đầu đo được sẽ cao hơn mức $\ln(7)$.
quá khớp 20 mẫu (overfit a small batch):
Ý nghĩa: Đây là một bước kiểm tra (sanity check) rất quan trọng để đảm bảo code không có bug. Nếu mô hình có thể học thuộc lòng (overfit) một tập dữ liệu cực nhỏ (20 mẫu) và đưa loss về gần 0, điều đó chứng tỏ toàn bộ luồng chạy của mô hình bao gồm: tính toán forward, tính loss, đạo hàm (backward) và cập nhật trọng số (optimizer) đều đã được cài đặt chính xác và hoạt động trơn tru.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [7]:
# Chạy baseline vài seed để xem đường cong và tính độ nhiễu
cfg_baseline = {**DEFAULT_CFG, "lr": 0.05, "epochs": 20, "batch": 512, "group": "baseline", "description": "Baseline SGD momentum"}
baseline_results = []

for s in [1, 2, 3]:
    cfg = {**cfg_baseline, "seed": s, "exp_id": f"base-s{s}"}
    res = run_experiment(cfg, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")
    baseline_results.append(res)
    print(f"Seed {s}: Val Macro-F1 = {res['summary']['val_macro_f1']:.4f}, Acc = {res['summary']['val_acc']:.4f}")

val_f1s = [r['summary']['val_macro_f1'] for r in baseline_results]
print(f"\nTrung bình Macro-F1: {np.mean(val_f1s):.4f} ± {np.std(val_f1s):.4f}")


Seed 1: Val Macro-F1 = 0.8314, Acc = 0.8988
Seed 2: Val Macro-F1 = 0.8442, Acc = 0.9041
Seed 3: Val Macro-F1 = 0.8369, Acc = 0.8997

Trung bình Macro-F1: 0.8375 ± 0.0052


**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

Hình dạng đường cong: Train loss giảm liên tục nhưng Val loss có xu hướng đi ngang/tăng nhẹ ở các epoch cuối, cho thấy mô hình bắt đầu bị quá khớp (overfitting).
Val_acc vs Đoán đa số: Độ chính xác đạt ~90%, vượt trội hoàn toàn so với mức ~50% của việc "đoán mò" theo class chiếm đa số. Mô hình thực sự học được đặc trưng tốt.
Độ nhiễu: Độ lệch chuẩn rất nhỏ (± 0.0052), chứng tỏ quá trình huấn luyện rất ổn định, ít bị ảnh hưởng bởi việc đổi random seed.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

Thí nghiệm opt-adamw — chủ đề optimizer

Dự đoán trước (viết TRƯỚC khi chạy): AdamW thường hội tụ nhanh hơn và tối ưu tốt hơn SGD thông thường. Dự đoán mô hình sẽ học nhanh hơn ở các epoch đầu và đạt Val Macro-F1 cao hơn (hoặc tương đương) so với baseline. Lưu ý learning rate được giảm xuống 0.001 (mức chuẩn của Adam) thay vì 0.05 như SGD.Thí nghiệm opt-adamw — chủ đề optimizer

Dự đoán trước (viết TRƯỚC khi chạy): AdamW thường hội tụ nhanh hơn và tối ưu tốt hơn SGD thông thường. Dự đoán mô hình sẽ học nhanh hơn ở các epoch đầu và đạt Val Macro-F1 cao hơn (hoặc tương đương) so với baseline. Lưu ý learning rate được giảm xuống 0.001 (mức chuẩn của Adam) thay vì 0.05 như SGD.

In [8]:
# Thí nghiệm: thay đổi Optimizer sang AdamW
cfg_adamw = {**DEFAULT_CFG, "optimizer": "adamw", "lr": 0.001, "epochs": 20, "batch": 512, 
             "exp_id": "opt-adamw", "group": "optimizer", "description": "AdamW"}
res_adamw = run_experiment(cfg_adamw, data)
save_result(res_adamw, f"{OUT_DIR}/results")
plot_run(res_adamw, f"{OUT_DIR}/figures/{cfg_adamw['exp_id']}.png")
print(f"AdamW: Val Macro-F1 = {res_adamw['summary']['val_macro_f1']:.4f}")


AdamW: Val Macro-F1 = 0.8432


**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

Đối chiếu sau khi chạy: Thực tế Val Macro-F1 đạt 0.8432, cao hơn so với mức trung bình của baseline SGD (~0.8375). Dự đoán chính xác, AdamW cho kết quả tốt hơn trong cùng số epoch (20 epochs).v

In [9]:
# Compare optimizer
plot_compare([baseline_results[0], res_adamw], "val_macro_f1", f"{OUT_DIR}/figures/compare_optimizer_f1.png", "Compare Optimizer F1")
plot_compare([baseline_results[0], res_adamw], "val_loss", f"{OUT_DIR}/figures/compare_optimizer_loss.png", "Compare Optimizer Loss")


## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [11]:
cfg_final = cfg_adamw
final_eval(cfg_final, res_adamw, data, f"{OUT_DIR}/predictions_eval.csv")

# Chạy chấm điểm
subprocess.run([sys.executable, "scripts/evaluate.py", 
                "--pred", f"{REPO_ROOT}/predictions_eval.csv", 
                "--out", f"{REPO_ROOT}/eval_result.json"], 
               cwd=REPO_ROOT, check=True)

with open(f"{REPO_ROOT}/eval_result.json") as f:
    eval_res = json.load(f)


Lưu file dự đoán thành công tại: ../predictions_eval.csv
Để chấm điểm, hãy chạy lệnh: python scripts/evaluate.py --pred ../predictions_eval.csv --out eval_result.json
n_eval = 116203
accuracy = 0.9006
macro_f1 = 0.8413   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9148  0.8860  0.9001
  1    56661     0.9066  0.9293  0.9178
  2     7151     0.8291  0.9371  0.8798
  3      549     0.8258  0.7687  0.7962
  4     1899     0.7335  0.7651  0.7490
  5     3473     0.8466  0.6421  0.7303
  6     4102     0.9413  0.8913  0.9156

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  37536   4545     9    0    87    16   175
1   3057  52653   272    1   427   198    53
2      0    215  6701   54     7   174     0
3      0      0   118  422     0     9     0
4     32    374    33    0  1453     7     0
5      6    248   949   34     6  2230     0
6    402     43     0    0     1     0  3656

đã ghi /home/tlht/day01/K4-Track4-D

In [13]:
# Hiển thị class report
import pandas as pd
print("Classification Report:")
df_report = pd.DataFrame(eval_res["per_class"]).T
display(df_report)

print("Confusion Matrix:")
print(np.array(eval_res["confusion_matrix"]))


Classification Report:


,0,1,2,3,4,5,6
cls,0.000000,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000
support,42368.000000,56661.000000,7151.000000,549.000000,1899.000000,3473.000000,4102.000000
precision,0.914776,0.906591,0.829126,0.825832,0.733468,0.846621,0.941298
recall,0.885952,0.929264,0.937072,0.768670,0.765140,0.642096,0.891273
f1,0.900133,0.917787,0.879800,0.796226,0.748969,0.730309,0.915602


Confusion Matrix:
[[37536  4545     9     0    87    16   175]
 [ 3057 52653   272     1   427   198    53]
 [    0   215  6701    54     7   174     0]
 [    0     0   118   422     0     9     0]
 [   32   374    33     0  1453     7     0]
 [    6   248   949    34     6  2230     0]
 [  402    43     0     0     1     0  3656]]


**Phân tích lỗi (tự viết):** ...

Mất cân bằng dữ liệu: Class 0 và 1 chiếm đa số, khiến mô hình dự đoán tốt ở 2 class này nhưng kém hẳn ở các class thiểu số (3, 4, 5).
Lỗi nặng nhất ở Class 5: Recall của Class 5 thấp nhất (0.64) do có tới 949 mẫu bị nhận diện nhầm sang Class 2. Hai class này có thể mang đặc trưng quá giống nhau.
Thiên vị class số đông: Mô hình có xu hướng hay đoán nhầm các class khác thành Class 1 (do Class 1 đông nhất). Ví dụ: Class 0 và Class 4 bị nhận diện nhầm sang Class 1 khá nhiều.
2:16 PMvMất cân bằng dữ liệu: Class 0 và 1 chiếm đa số, khiến mô hình dự đoán tốt ở 2 class này nhưng kém hẳn ở các class thiểu số (3, 4, 5).
Lỗi nặng nhất ở Class 5: Recall của Class 5 thấp nhất (0.64) do có tới 949 mẫu bị nhận diện nhầm sang Class 2. Hai class này có thể mang đặc trưng quá giống nhau.
Thiên vị class số đông: Mô hình có xu hướng hay đoán nhầm các class khác thành Class 1 (do Class 1 đông nhất). Ví dụ: Class 0 và Class 4 bị nhận diện nhầm sang Class 1 khá nhiều.
2:16 PM

In [14]:
# Cập nhật kết quả ra file Excel
results = load_results(f"{OUT_DIR}/results")
rows = [to_row(r) for r in results]
# Đổ thêm eval metrics cho dòng final
for row in rows:
    if row["exp_id"] == cfg_final["exp_id"]:
        row["eval_acc"] = eval_res["accuracy"]
        row["eval_macro_f1"] = eval_res["macro_f1"]

write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
print(f"Saved to {OUT_DIR}/experiments.xlsx")


Saved to ../experiments.xlsx
